In [9]:
import numpy as np
from scipy.sparse import csr_matrix
import time

data = np.load('user_movie_rating.npy')

In [10]:
users= data[:,0]
movies = data[:, 1]
ratings = data [:, 2]

print("Length of users:", len(users))
print("Length of movies:", len(movies))
print("Length of ratings:", len(ratings))
print(len(np.ones_like(users)))

Length of users: 65225506
Length of movies: 65225506
Length of ratings: 65225506
65225506


In [11]:
movie_user_matrix=csr_matrix((np.ones_like(users-1), (movies-1, users-1)))

In [12]:
print("Sparse matrix shape (movies x users):", movie_user_matrix.shape)

Sparse matrix shape (movies x users): (17770, 103703)


In [19]:
#User 1 rates movie 30
movie_user_matrix[29,0]

1

In [14]:
def minhash_signature(sparse_matrix, num_permutations):
    """
    Compute minhash signatures using vectorized operations
    
    Parameters:
    - sparse_matrix (csr_matrix): sparse movie-user matrix where rows represent movies, columns represent users.
    - num_permutations (int): number of minhash permutations (hash functions) to apply

    Returns:
    - np.ndarray: a 2D array of shape (num_permutations, num_users), where each column
      corresponds to a users minhash signature across all permutations
    """
    num_movies, num_users = sparse_matrix.shape

    #Initialize signature matrix
    signatures = np.full((num_permutations, num_users), np.inf, dtype=np.float64)

    #Generate random hash functions using parameters (a, b)
    rng = np.random.default_rng(seed=42)  #Fixed seed for reproducibility
    a = rng.integers(1, high=num_movies, size=num_permutations, dtype=np.int64)
    b = rng.integers(0, high=num_movies, size=num_permutations, dtype=np.int64)
    p = 2**31 - 1  #A large prime number for hashing

    #Select the non zero elements
    row_indices, col_indices = sparse_matrix.nonzero()  #Get row and column indices of non-zero elements

    for row, col in zip(row_indices, col_indices):
        #Compute hash values for the current row across all hash functions
        hash_values = (a * row + b) % p
        #update the signatures for the column (user) with the minimum hash values
        signatures[:, col] = np.minimum(signatures[:, col], hash_values)

    return signatures

In [15]:
import time
t0 = time.time()
test  = minhash_signature(movie_user_matrix,300)
t1 = time.time()
time=t1-t0
print(time)

601.578999042511


In [16]:
print(test)
print(test.max())
print(test.min())

[[  52456.   17564.   52456. ...  192024.   17564.   49284.]
 [ 413520.  110954.  413520. ... 1623784.  110954.  386014.]
 [ 337504.   81600.  337504. ... 1361120.   81600.  314240.]
 ...
 [ 240051.   68451.  240051. ...  926451.   68451.  224451.]
 [ 157809.   46555.  157809. ...  602825.   46555.  147695.]
 [  22581.   14111.   22581. ...   56461.   14111.   21811.]]
13287853.0
94.0
